In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict


In [ ]:
# --------------------------
# Global simulation constants
# --------------------------
DT = 0.05  # s

# FIA MGU-K constraints
MGUK_MAX_DEPLOY_POWER = 120_000.0   # W
MGUK_MAX_DEPLOY_ENERGY_LAP = 4e6    # J
MGUK_MAX_REGEN_POWER = 120_000.0    # W (early-stage assumption)

# Battery model
BATTERY_CAPACITY_J = 4e6            # J
SOC_MIN, SOC_MAX = 0.20, 1.00
INITIAL_SOC = 0.78

# Vehicle/track physics parameters (calibration targets for modern F1-like behavior)
VEHICLE_MASS = 798.0                # kg
G = 9.81                            # m/s^2
RHO_AIR = 1.225                     # kg/m^3
CD_A = 1.00                         # effective drag area (reduced from prior version)
C_RR = 0.012                        # rolling resistance coefficient
TRACTION_FORCE_LIMIT = 17_500.0     # N
ICE_WHEEL_POWER = 650_000.0         # W
MAX_BRAKE_DECEL = 5.8 * G           # m/s^2 cap to avoid non-physical braking spikes
MIN_SPEED_MPS = 35.0 / 3.6          # m/s, prevent unrealistic near-stop behavior

# Hybrid conversion efficiencies
DEPLOY_EFFICIENCY = 0.95
REGEN_EFFICIENCY = 0.70


In [ ]:
@dataclass(frozen=True)
class TrackSegment:
    name: str
    segment_type: str  # "accel", "brake", "coast"
    length_m: float
    v_entry_kph: float
    v_exit_kph: float
    deployment_priority: float
    regen_priority: float
    overtaking_relevance: float
    notes: str

    @property
    def v_entry_mps(self) -> float:
        return self.v_entry_kph / 3.6

    @property
    def v_exit_mps(self) -> float:
        return self.v_exit_kph / 3.6


def build_gilles_villeneuve_track() -> List[TrackSegment]:
    """Segmented Circuit Gilles Villeneuve model with strategy metadata."""
    return [
        TrackSegment("Start/Finish Straight", "accel", 780, 145, 305, 0.70, 0.05, 0.45, "Medium-value deployment before T1."),
        TrackSegment("Turn 1-2 Chicane Braking", "brake", 210, 305, 125, 0.05, 0.80, 0.40, "Heavy regen zone."),
        TrackSegment("T2 Exit to T3 Approach", "accel", 520, 125, 285, 0.60, 0.05, 0.25, "Moderate deployment value."),
        TrackSegment("Turn 3-4 Chicane Braking", "brake", 180, 285, 118, 0.05, 0.65, 0.20, "Moderate-heavy braking."),
        TrackSegment("Casino Straight", "accel", 820, 118, 298, 0.75, 0.05, 0.50, "Strong deployment value."),
        TrackSegment("L'Epingle Hairpin Braking (T10)", "brake", 220, 298, 86, 0.05, 1.00, 0.85, "Strongest regen + overtaking setup."),
        TrackSegment("Back Straight (T10 Exit to T13)", "accel", 1080, 86, 330, 1.00, 0.05, 1.00, "Primary deployment target."),
        TrackSegment("Wall of Champions Braking (T13-14)", "brake", 190, 330, 145, 0.05, 0.85, 0.70, "Heavy final braking."),
    ]


def validate_track(track: List[TrackSegment]) -> None:
    valid_types = {"accel", "brake", "coast"}
    if len(track) == 0:
        raise ValueError("Track cannot be empty.")

    for i, seg in enumerate(track):
        if seg.segment_type not in valid_types:
            raise ValueError(f"Invalid segment type in {seg.name}")
        if seg.length_m <= 0:
            raise ValueError(f"Invalid length in {seg.name}")
        for field_name, value in (
            ("deployment_priority", seg.deployment_priority),
            ("regen_priority", seg.regen_priority),
            ("overtaking_relevance", seg.overtaking_relevance),
        ):
            if not (0.0 <= value <= 1.0):
                raise ValueError(f"{field_name} out of range for {seg.name}")

        if i > 0:
            continuity_gap = abs(track[i - 1].v_exit_kph - seg.v_entry_kph)
            if continuity_gap > 7.5:
                raise ValueError(f"Speed continuity mismatch between {track[i - 1].name} and {seg.name}")


track = build_gilles_villeneuve_track()
validate_track(track)
print(f"Track loaded: {len(track)} segments, {sum(s.length_m for s in track):.0f} m")


In [ ]:
def deployment_policy(segment: TrackSegment, soc: float, deploy_used_j: float) -> float:
    """Track-aware MGU-K deployment policy (simple baseline controller)."""
    if segment.segment_type != "accel":
        return 0.0
    if soc <= SOC_MIN + 0.03:
        return 0.0

    energy_left = MGUK_MAX_DEPLOY_ENERGY_LAP - deploy_used_j
    if energy_left <= 0:
        return 0.0

    # Higher-priority zones (like back straight) get stronger deployment.
    soc_factor = np.clip((soc - SOC_MIN) / (SOC_MAX - SOC_MIN), 0.0, 1.0)
    priority_factor = 0.40 + 0.60 * segment.deployment_priority
    p_cmd = MGUK_MAX_DEPLOY_POWER * soc_factor * priority_factor

    return float(np.clip(p_cmd, 0.0, min(MGUK_MAX_DEPLOY_POWER, energy_left / DT)))


In [ ]:
def resistive_forces(speed_mps: float) -> float:
    drag = 0.5 * RHO_AIR * CD_A * speed_mps**2
    rolling = C_RR * VEHICLE_MASS * G
    return drag + rolling


def run_lap(track: List[TrackSegment], ers_enabled: bool = True) -> Dict[str, np.ndarray]:
    soc = INITIAL_SOC
    deploy_used = 0.0
    regen_harvested = 0.0

    t = 0.0
    v = track[0].v_entry_mps

    time_trace, soc_trace = [], []
    deploy_power_trace, regen_power_trace, speed_trace = [], [], []

    for segment in track:
        s_local = 0.0
        v = max(MIN_SPEED_MPS, segment.v_entry_mps)

        while s_local < segment.length_m:
            f_resist = resistive_forces(v)
            p_deploy = 0.0
            p_regen = 0.0

            # Distance-based target speed interpolation inside each segment.
            progress = np.clip(s_local / max(segment.length_m, 1e-6), 0.0, 1.0)
            v_target = segment.v_entry_mps + progress * (segment.v_exit_mps - segment.v_entry_mps)

            if segment.segment_type == "accel":
                f_ice = min(ICE_WHEEL_POWER / max(v, 1.0), TRACTION_FORCE_LIMIT)

                if ers_enabled:
                    p_deploy = deployment_policy(segment, soc, deploy_used)
                f_mguk = (p_deploy * DEPLOY_EFFICIENCY) / max(v, 1.0)

                # PD-like controller to avoid unstable speed jumps.
                a_ctrl = 0.8 * (v_target - v)
                a = np.clip((f_ice + f_mguk - f_resist) / VEHICLE_MASS + a_ctrl, -2.0, 12.0)

            elif segment.segment_type == "brake":
                v_err = v - v_target
                a_des = np.clip(-0.8 * v_err - 1.5, -MAX_BRAKE_DECEL, -0.3)

                required_brake_force = max(0.0, VEHICLE_MASS * abs(a_des) + f_resist)
                recoverable_brake_force = 0.62 * required_brake_force

                if ers_enabled and soc < SOC_MAX - 0.01:
                    p_regen_theoretical = recoverable_brake_force * v
                    soc_acceptance = np.clip((SOC_MAX - soc) / 0.25, 0.0, 1.0)
                    p_regen = min(p_regen_theoretical, MGUK_MAX_REGEN_POWER) * soc_acceptance
                    e_harv = p_regen * REGEN_EFFICIENCY * DT
                    regen_harvested += e_harv
                    soc += e_harv / BATTERY_CAPACITY_J

                a = a_des
            else:
                a = -f_resist / VEHICLE_MASS

            if p_deploy > 0.0:
                e_dep = p_deploy * DT
                deploy_used += e_dep
                soc -= e_dep / BATTERY_CAPACITY_J

            soc = float(np.clip(soc, SOC_MIN, SOC_MAX))
            v = float(np.clip(v + a * DT, MIN_SPEED_MPS, 95.0))
            s_step = v * DT
            s_local += s_step
            t += DT

            time_trace.append(t)
            soc_trace.append(soc)
            deploy_power_trace.append(p_deploy)
            regen_power_trace.append(p_regen)
            speed_trace.append(v)

    return {
        "lap_time_s": t,
        "deploy_used_j": deploy_used,
        "regen_harvested_j": regen_harvested,
        "time": np.array(time_trace),
        "soc": np.array(soc_trace),
        "deploy_power": np.array(deploy_power_trace),
        "regen_power": np.array(regen_power_trace),
        "speed": np.array(speed_trace),
    }


In [ ]:
def quality_checks(result: Dict[str, np.ndarray], label: str) -> Dict[str, float]:
    soc = result["soc"]
    speed_kph = result["speed"] * 3.6
    deploy = result["deploy_power"]
    regen = result["regen_power"]

    checks = {
        "soc_min": float(np.min(soc)),
        "soc_max": float(np.max(soc)),
        "speed_min_kph": float(np.min(speed_kph)),
        "speed_max_kph": float(np.max(speed_kph)),
        "deploy_cap_violation_w": float(np.max(np.maximum(deploy - MGUK_MAX_DEPLOY_POWER, 0.0))),
        "regen_cap_violation_w": float(np.max(np.maximum(regen - MGUK_MAX_REGEN_POWER, 0.0))),
    }

    print(f"\n--- Quality checks: {label} ---")
    print(f"SOC range          : {checks['soc_min']:.3f} to {checks['soc_max']:.3f}")
    print(f"Speed range        : {checks['speed_min_kph']:.1f} to {checks['speed_max_kph']:.1f} km/h")
    print(f"Deploy cap overrun : {checks['deploy_cap_violation_w']:.2f} W")
    print(f"Regen cap overrun  : {checks['regen_cap_violation_w']:.2f} W")
    return checks


def moving_average(x: np.ndarray, window: int = 25) -> np.ndarray:
    if window <= 1:
        return x.copy()
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="same")


result_baseline = run_lap(track, ers_enabled=False)
result_ers = run_lap(track, ers_enabled=True)

lap_time_baseline = result_baseline["lap_time_s"]
lap_time_ers = result_ers["lap_time_s"]
delta_lap = lap_time_baseline - lap_time_ers

print("=== Simulation Summary (Physics-Based Lap Integration) ===")
print(f"Baseline lap time (ERS OFF): {lap_time_baseline:.2f} s")
print(f"Hybrid lap time   (ERS ON) : {lap_time_ers:.2f} s")
print(f"Estimated lap-time gain    : {delta_lap:.2f} s")
print()
print(f"Energy deployed : {result_ers['deploy_used_j']/1e6:.2f} MJ / 4.00 MJ")
print(f"Energy harvested: {result_ers['regen_harvested_j']/1e6:.2f} MJ")
print(f"Final SOC       : {result_ers['soc'][-1]:.3f}")

quality_checks(result_baseline, "ERS OFF")
quality_checks(result_ers, "ERS ON")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(result_ers["time"], result_ers["soc"], label="SOC")
plt.title("Battery SOC vs Time - Circuit Gilles Villeneuve")
plt.xlabel("Time (s)")
plt.ylabel("SOC")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Raw and smoothed power traces for easier interpretation.
deploy_kw = result_ers["deploy_power"] / 1000.0
regen_kw = result_ers["regen_power"] / 1000.0

plt.figure(figsize=(10, 5))
plt.plot(result_ers["time"], deploy_kw, alpha=0.35, label="Deploy Power Raw (kW)")
plt.plot(result_ers["time"], regen_kw, alpha=0.35, label="Regen Power Raw (kW)")
plt.plot(result_ers["time"], moving_average(deploy_kw, 25), linewidth=2.2, label="Deploy Power Smoothed (kW)")
plt.plot(result_ers["time"], moving_average(regen_kw, 25), linewidth=2.2, label="Regen Power Smoothed (kW)")
plt.title("MGU-K Power vs Time")
plt.xlabel("Time (s)")
plt.ylabel("Power (kW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(result_ers["time"], result_ers["speed"] * 3.6)
plt.title("Vehicle Speed Trace")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.grid(True, alpha=0.3)
plt.show()
